In [1]:
from deepeval.models import DeepEvalBaseLLM
from pydantic import BaseModel
import instructor
from openai import OpenAI

class CustomOllamaLlama3_1(DeepEvalBaseLLM):
    def __init__(self):
        self.model_name = 'llama3.1'
        self.client = instructor.from_openai(
            OpenAI(
                base_url="http://localhost:11434/v1",
                api_key="ollama",  # required, but unused
            ),
            mode=instructor.Mode.JSON,
        )

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str, schema: BaseModel) -> BaseModel:
        model_name = self.load_model()
        response = self.client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            response_model=schema,
        )
        return response

    async def a_generate(self, prompt: str, schema: BaseModel) -> BaseModel:
        return self.generate(prompt, schema)

    def get_model_name(self):
        return "Ollama LLaMA 3.1 8B"

In [2]:
from pydantic import BaseModel
import asyncio

class JokeSchema(BaseModel):
    joke: str

# custom_llm = CustomOllamaLlama3_1()
custom_llm = "gpt-4o-mini"

prompt = "Tell me a joke."

In [102]:
# Synchronous
generated_response = custom_llm.generate(prompt=prompt, schema=JokeSchema)
print(generated_response.model_dump_json(indent=2))

# Async
async def main():
    async_response = await custom_llm.a_generate(prompt=prompt, schema=JokeSchema)
    print(async_response.model_dump_json(indent=2))

asyncio.run(main())

INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


{
  "joke": "A man walks into a library and asks the librarian, 'Do you have any books on Pavlov's dogs and Schrödinger's cat?' The librarian replies, 'It rings a bell, but I'm not sure if it's here or not.'"
}


INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


{
  "joke": "Why don't scientists trust atoms? Because they make up everything!"
}


In [3]:
from deepeval.metrics import (AnswerRelevancyMetric, GEval, FaithfulnessMetric, ContextualPrecisionMetric, 
ContextualRecallMetric, ContextualRelevancyMetric, HallucinationMetric, BiasMetric, ToxicityMetric,
ConversationCompleteness, ConversationRelevancy, KnowledgeRetentionMetric)
from deepeval.test_case import LLMTestCaseParams


answer_relevancy_metric = AnswerRelevancyMetric(model=custom_llm)
correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the expected output.",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'expected output'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=custom_llm
)
faithfulness_metric = FaithfulnessMetric(model=custom_llm)
contextual_precision_metric = ContextualPrecisionMetric(model=custom_llm)
contextual_recall_metric = ContextualRecallMetric(model=custom_llm)
contextual_relevancy_metric = ContextualRelevancyMetric(model=custom_llm)
hallucination_metric = HallucinationMetric(model=custom_llm)
bias_metric = BiasMetric(model=custom_llm)
toxicity_metric = ToxicityMetric(model=custom_llm)
conversation_completeness_metric = ConversationCompleteness(model=custom_llm)
conversation_relevancy_metric = ConversationRelevancy(model=custom_llm)
knowledge_retention_metric = KnowledgeRetentionMetric(model=custom_llm)

metrics = [
    answer_relevancy_metric,
    correctness_metric,
    faithfulness_metric,
    contextual_precision_metric,
    contextual_recall_metric,
    contextual_relevancy_metric,
    hallucination_metric,
    bias_metric,
    toxicity_metric,
    conversation_completeness_metric,
    conversation_relevancy_metric,
    knowledge_retention_metric
]

In [6]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  expected_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."],
  context=["Joe Biden is the current president of the United States of America."]
)

In [4]:
original_dataset = [
    {
        "input": "Mengapa perlu membentuk Undang-Undang baru tentang Provinsi Jawa Barat?",
        "expected_output": "Undang-Undang Nomor 11 Tahun 1950 tentang Pembentukan Propinsi Djawa Barat sudah tidak sesuai lagi dengan perkembangan hukum sehingga perlu diganti.",
        "context": [
            "bahwa Undang-Undang Nomor 11 Tahun 1950 tentang Pembentukan Propinsi Djawa Barat sudah tidak sesuai lagi dengan perkembangan hukum sehingga perlu diganti;",
            "bahwa berdasarkan pertimbangan sebagaimana dimaksud dalam huruf a, huruf b, dan huruf c, perlu membentuk Undang-Undang tentang Provinsi Jawa Barat."
        ],
    },
    {
        "input": "Apa tujuan dari pembentukan provinsi di Indonesia?",
        "expected_output": "Provinsi dibentuk untuk meningkatkan penyelenggaraan pemerintahan dan pembangunan yang efektif sesuai amanat Undang-Undang Dasar Negara Republik Indonesia Tahun 1945.",
        "context": [
            "bahwa provinsi merupakan salah satu daerah di wilayah Negara Kesatuan Republik Indonesia yang dibentuk untuk meningkatkan penyelenggaraan pemerintahan dan pembangunan yang efektif sesuai amanat Undang-Undang Dasar Negara Republik Indonesia Tahun 1945;"
        ],
    },
    {
        "input": "Kapan tanggal pembentukan Provinsi Jawa Barat?",
        "expected_output": "Tanggal 4 Juli 1950 merupakan tanggal pembentukan Provinsi Jawa Barat berdasarkan Undang-Undang Nomor 11 Tahun 1950 tentang Pembentukan Propinsi Djawa Barat.",
        "context": [
            "Tanggal 4 Juli 1950 merupakan tanggal pembentukan Provinsi Jawa Barat berdasarkan Undang-Undang Nomor 11 Tahun 1950 tentang Pembentukan Propinsi Djawa Barat."
        ],
    },
    {
        "input": "Berapa jumlah kabupaten dan kota yang ada di Provinsi Jawa Barat?",
        "expected_output": "Provinsi Jawa Barat terdiri atas 18 kabupaten dan 9 kota.",
        "context": [
            "Provinsi Jawa Barat terdiri atas 18 kabupaten dan 9 (sembilan) kota, yaitu:",
            "a. Kabupaten Bogor;",
            "b. Kabupaten Sukabumi;",
            "c. Kabupaten Cianjur;",
            "d. Kabupaten Bandung;",
            "e. Kabupaten Garut;",
            "f. Kabupaten Tasikmalaya;",
            "g. Kabupaten Ciamis;",
            "h. Kabupaten Kuningan;",
            "i. Kabupaten Cirebon;",
            "j. Kabupaten Majalengka;",
            "k. Kabupaten Sumedang;",
            "l. Kabupaten Indramayu;",
            "m. Kabupaten Subang;",
            "n. Kabupaten Purwakarta;",
            "o. Kabupaten Karawang;",
            "p. Kabupaten Bekasi;",
            "q. Kabupaten Bandung Barat;",
            "r. Kabupaten Pangandaran;",
            "s. Kota Bogor;",
            "t. Kota Sukabumi;",
            "u. Kota Bandung;",
            "v. Kota Cirebon;",
            "w. Kota Bekasi;",
            "x. Kota Depok;",
            "y. Kota Cimahi;",
            "z. Kota Tasikmalaya;",
            "aa. Kota Banjar."
        ],
    },
    {
        "input": "Dimana ibu kota Provinsi Jawa Barat?",
        "expected_output": "Ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung.",
        "context": [
            "Ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung."
        ],
    }
]

In [7]:
# from utils.rag.open_source.basic_agent import caller_with_sources
from utils.rag.basic_agent import caller_with_sources
from utils.utils import generate_id
from deepeval.test_case import LLMTestCase

test_cases = []
for datapoint in original_dataset:
    input = datapoint.get("input", None)
    actual_output, retrieval_context_documents = caller_with_sources(input, generate_id())
    expected_output = datapoint.get("expected_output", None)
    context = datapoint.get("context", None)
    retrieval_context = [doc.page_content for doc in retrieval_context_documents]

    test_case = LLMTestCase(
        input=input,
        actual_output=actual_output,
        expected_output=expected_output,
        context=context,
        retrieval_context=retrieval_context
    )
    test_cases.append(test_case)

In [11]:
from deepeval.dataset import EvaluationDataset

dataset = EvaluationDataset(test_cases=test_cases)
dataset.evaluate(metrics=metrics)

Output()

Evaluating test cases...
Event loop is already running. Applying nest_asyncio patch to allow async execution...


Output()

Output()

Output()

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the response directly addresses the need for a new law regarding West Java without any irrelevant statements., error: None)
  - ✅ Correctness (GEval) (score: 0.8622459331201855, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The actual output provides a clear rationale for the need to form a new law, citing the inadequacy of the existing law and outlining specific goals like legal synchronization and improved governance, but lacks some specific historical context or examples of the law's shortcomings., error: None)
  - ✅ Faithfulness (score: 0.875, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.88 because the actual output implies the necessity for new legislation regarding Provinsi Jawa Barat, contradicting the retrieval context which states that Undang-Undang Republik Indones

Exception: Limit Reached: Projects on the FREE plan cannot log more than 10 test runs a day. Please upgrade your project plan at https://app.confident-ai.com/project/clztlbe0v008e2eq8vxcuzp44/project-details to continue.